# 05 — Target Label Construction

Notebook này xây dựng 2 biến mục tiêu:
- **`will_return_within_30_days`** — Classification label (0/1)
- **`days_until_next_purchase`** — Regression label (số ngày)

Logic: tính từ **giao dịch cuối** của mỗi khách đến **lần mua kế tiếp** (nếu có).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT  = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

transactions = pd.read_csv(PROCESSED_DIR / 'transactions_enriched.csv')
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], errors='coerce')

print('transactions shape:', transactions.shape)
print(transactions[['customer_id', 'transaction_date', 'total_amount']].head())

## Buoc 1 — Snapshot Date

Snapshot date = ngay sau giao dich cuoi cung trong dataset.
Tat ca tinh toan khoang cach deu tinh den moc nay.

In [ ]:
snapshot_date = transactions['transaction_date'].max() + pd.Timedelta(days=1)
print('Snapshot date:', snapshot_date)

## Buoc 2 — Tinh days_until_next_purchase

Sort giao dich theo thoi gian, dung `shift(-1)` lay ngay giao dich ke tiep.
Chi giu **dong cuoi cung** cua moi khach (last transaction -> next purchase).

In [ ]:
tx_sorted = transactions.sort_values(['customer_id', 'transaction_date'])

# next_purchase_date = giao dich lien ke sau cua cung khach
tx_sorted['next_purchase_date'] = (
    tx_sorted.groupby('customer_id')['transaction_date'].shift(-1)
)

# Lay dong cuoi cua moi khach
last_tx = (
    tx_sorted.groupby('customer_id').last().reset_index()
    [['customer_id', 'transaction_date', 'next_purchase_date']]
    .rename(columns={'transaction_date': 'last_purchase_date'})
)

# Tinh so ngay den lan mua ke tiep
last_tx['days_until_next_purchase'] = (
    last_tx['next_purchase_date'] - last_tx['last_purchase_date']
).dt.days

print(last_tx.head(10).to_string())

## Buoc 3 — Xay dung will_return_within_30_days

Khach duoc gan nhan **1** neu co giao dich ke tiep **trong vong 30 ngay** sau lan mua cuoi.

In [ ]:
N_DAYS = 30

last_tx['will_return_within_30_days'] = (
    last_tx['days_until_next_purchase'].notna() &
    (last_tx['days_until_next_purchase'] <= N_DAYS)
).astype(int)

n_total    = len(last_tx)
n_positive = last_tx['will_return_within_30_days'].sum()
n_censored = last_tx['days_until_next_purchase'].isna().sum()

print(f'Tong khach:                  {n_total}')
print(f'Positive (mua lai <=30 ngay): {n_positive} ({n_positive/n_total*100:.1f}%)')
print(f'Censored (chua mua lai):      {n_censored} ({n_censored/n_total*100:.1f}%)')
print('\nLabel distribution:')
print(last_tx['will_return_within_30_days'].value_counts())

## Buoc 4 — Xu ly Censored Data

**Censored** = khach khong co giao dich ke tiep tai thoi diem snapshot. Gom 2 loai:

| Loai | Y nghia | Xu ly |
|---|---|---|
| Mua 1 lan duy nhat | Khong co lan ke tiep trong lich su | `will_return = 0`, `days = NaN` |
| Mua nhieu lan, lan cuoi gan snapshot | Chua du thoi gian de quay lai | `will_return = 0`, `days = NaN` |

**Classification:** Censored = 0 (conservative assumption).  
**Regression:** Loai censored khoi tap train, chi train tren khach **co** `days_until_next_purchase`.

In [ ]:
# Khach mua dung 1 lan
n_one_time = (transactions.groupby('customer_id')['transaction_id'].count() == 1).sum()
print(f'Khach mua dung 1 lan:         {n_one_time}')

# Censored multi-buyer
tx_count = transactions.groupby('customer_id')['transaction_id'].count().rename('tx_count').reset_index()
multi_censored = last_tx[last_tx['days_until_next_purchase'].isna()].merge(tx_count, on='customer_id')
print(f'Censored do gan snapshot:     {(multi_censored["tx_count"] > 1).sum()}')

# Subset cho regression
regression_df = last_tx[last_tx['days_until_next_purchase'].notna()].copy()
print(f'\nTap regression (co next purchase): {len(regression_df)} khach')

## Buoc 5 — Export target labels

In [ ]:
target_labels = last_tx[[
    'customer_id',
    'last_purchase_date',
    'days_until_next_purchase',
    'will_return_within_30_days'
]]

target_labels.to_csv(PROCESSED_DIR / 'target_labels.csv', index=False)
print('Saved target_labels.csv:', target_labels.shape)
print(target_labels.head())